# Press-Fit Envelope Curves Calculator

## Project Overview
The Press-Fit Envelope Curves Calculator is a Python-based web application developed to automate the generation of process control envelopes for Press-Fit manufacturing operations. The tool was designed to run directly in Google Colab, eliminating the need for software installation and enabling easy deployment across manufacturing and quality teams.

The application imports Press-Fit force-displacement curve data from Excel (.xlsx) or CSV files, preprocesses and standardizes the datasets, calculates statistical envelope limits, and provides an interactive environment for curve validation and analysis.

By automating a traditionally manual and time-consuming task, the application improves consistency, reduces engineering effort, and facilitates data-driven process validation.

## Objectives

- Automate the generation of Press-Fit envelope curves.
- Standardize curve processing and analysis.
- Reduce manual calculations and engineering lead time.
- Improve process validation consistency across production lines.
- Provide a user-friendly interface that requires no programming knowledge.

## Key Features

### Data Import

The application supports:

- Excel files (.xlsx)
- CSV files (.csv)

Data can be uploaded directly from a local computer through the graphical interface.

### Data Preprocessing

Once the files are loaded, the software:

- Extracts force and displacement measurements.
- Converts and formats raw data into a standardized structure.
- Validates data integrity before processing.

### Curve Interpolation

Since individual Press-Fit curves often contain different numbers of measurement points, the software performs interpolation to align all curves onto a common distance scale.

This enables accurate point-by-point statistical analysis across the entire dataset.

### Envelope Curve Calculation

The application calculates upper and lower envelope limits based on a user-defined sigma level.

The user can adjust:

- Resolution: Number of interpolation points used for each curve.
- Sigma Value: Standard deviation multiplier used to generate the envelope limits.
- Envelope Point Count: Number of points included in the final envelope table.

### Interactive Visualization

An interactive plot allows users to:

- Visualize all imported curves.
- Review the calculated envelope limits.
- Dynamically update calculation parameters.
- Evaluate process behavior before exporting results.

### Curve Filtering

To improve envelope quality, users can:

- Remove individual curves identified as non-representative or generated from unsuccessful assemblies.
- Automatically remove the final point from all curves to eliminate measurement noise commonly observed at the end of Press-Fit cycles.

### Exportable Results

Once the analysis is complete, users can copy the generated envelope point table for implementation in production equipment or process validation documentation.

## Technical Implementation

### Development Environment

- Python
- Google Colab
- Pandas
- NumPy
- SciPy
- Plotly

### Processing Workflow
1. mport curve data from Excel or CSV files.
2. Extract force and displacement information.
3. Standardize and clean datasets.
4. Interpolate all curves to a common distance axis.
5. Calculate mean and standard deviation at each interpolation point.
6. Generate upper and lower envelope limits using the selected sigma level.
7. Display interactive visualizations.
8. Allow user-driven curve filtering and parameter adjustment.
9. Generate final envelope point tables.

### Design Considerations

To simplify usage for production and quality personnel, the entire application was implemented within a single executable code block in Google Colab. This design prevents execution-order errors and ensures that users only need to run the notebook once to access all functionalities.

## User Guide
### Running the Application
1. Open the Google Colab notebook.
2. Execute the main code cell by clicking the Run (▶) button or pressing Ctrl + Enter.
3. Scroll to the final section of the notebook.
4. Select whether the data corresponds to:
    - Production Curves
    - Minimum/Maximum Curves
5. Select the Press-Fit system type:
    - Kistler
    - Promess
6. Click Upload Files.
7. Select the corresponding Excel or CSV file.
8. Wait for the upload and processing to complete.
9. Review the interactive graph.
10. Adjust parameters as needed:
    - Resolution
    - Sigma value
    - Envelope point count
11. Optionally:
    - Remove invalid curves.
    - Remove the final point from all curves.
12. Copy the generated envelope table for use in process validation.
13. To perform a new analysis, restart the workflow from Step 1.

## Benefits
- Reduces manual engineering effort.
- Standardizes Press-Fit envelope generation.
- Improves repeatability and consistency.
- Supports data-driven process validation.
- Provides rapid analysis of large datasets.
- Enables interactive quality assessment and outlier identification.

## Contact Information

**Juan Pablo Ramírez Guerra**

 Process Engineer | Data Scientist

📧 juan.2.ramirez.guerra@aumovio.com

 📞 +52 33 3455 0363



In [ ]:
"""
Press-Fit Envelope Curve Calculator

This application imports force-displacement data from
Promess and Kistler Press-Fit systems, performs data
interpolation, calculates statistical envelope curves,
and provides interactive visualization and filtering
tools for process validation.

Author: Juan Pablo Ramírez Guerra
"""

# Enable support for third-party widgets in Google Colab
from google.colab import output

output.enable_custom_widget_manager()

# Import required libraries
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
import os
import shutil
import pandas as pd
from google.colab import files
from IPython.display import display, clear_output
from scipy.interpolate import interp1d

# Global variables
distances = []
forces = []
distances_min = []
forces_min = []
distances_max = []
forces_max = []


# Delete everything inside /content (files and folders)
def delete_everything_in_content():
    for item in os.listdir("/content"):
        item_path = os.path.join("/content", item)
        try:
            if os.path.isfile(item_path) or os.path.islink(item_path):
                os.remove(item_path)
            elif os.path.isdir(item_path):
                shutil.rmtree(item_path)
        except Exception as e:
            print(f"Error deleting {item_path}: {e}")


# Function to read data of xlsx file from Promess Press
def read_data_promess_xlsx(file_path):
    data = pd.read_excel(file_path, header=0)

    # Find the index of the column 'Value0'
    if "Value0" in data.columns:
        start_index = data.columns.get_loc("Value0")
        data = data.iloc[:, start_index:]  # Keep columns from 'Value0' onward
    elif "Val 1" in data.columns:
        start_index = data.columns.get_loc("Val 1")
        data = data.iloc[:, start_index:]  # Keep columns from 'Value0' onward
    else:
        print("'Value0' column not found in the Excel file.")

    num_rows = data.shape[0]
    num_curves = num_rows // 2
    distances = [data.iloc[i * 2, :].dropna().values for i in range(num_curves)]
    forces = [data.iloc[i * 2 + 1, :].dropna().values for i in range(num_curves)]

    # Erase duplicate curves
    unique = set()
    filtered_distances = []
    filtered_forces = []

    for d, f in zip(distances, forces):
        key = (tuple(d), tuple(f))
        if key not in unique:
            unique.add(key)
            filtered_distances.append(d)
            filtered_forces.append(f)

    # Update the original distance and force arrays
    distances = filtered_distances
    forces = filtered_forces

    print(len(distances))
    print(len(forces))
    return distances, forces


# Read Kistler curve data from CSV files
def read_data_kistler_csv(folder_path, delimeter, decimal_point):
    distances = []
    forces = []

    # Iterate through all CSV files in the selected folder
    for filename in os.listdir(folder_path):
        if filename.endswith(".csv"):
            file_path = os.path.join(folder_path, filename)
            file = open(file_path)
            lines = file.readlines()
            lines = lines[279:]
            lines = lines[: lines.index("\n")]

            parameters = []
            distance = []
            force = []

            if len(lines[0].split(delimeter)) != 4:
                if delimeter == ";":
                    delimeter = ","
                elif delimeter == ",":
                    delimeter = ";"
            for linea in lines:
                parameter = linea.split(delimeter)
                parameter = parameter[1:3]
                parameter = [x.replace(",", ".") for x in parameter]
                parameter = [float(x) for x in parameter]
                distance.append(parameter[0])
                force.append(parameter[1])
            distances.append(distance)
            forces.append(force)
    return distances, forces


# Erase the curve in distances and forces
def delete_curve_by_index(curve_index):
    global distances, forces

    # Index validation
    if curve_index < 0 or curve_index >= len(distances):
        print(f"Index out of range: {curve_index}")
        return

    # Eliminar la curva
    del distances[curve_index]
    del forces[curve_index]
    print(f"Curve {curve_index} was deleted successfully.")


# Interpolate curves onto a common distance axis
def interpolate_mixed(distances, forces, distance_axis):
    interpolated_forces = []
    for x, y in zip(distances, forces):
        split_index = int(len(distance_axis) * 0.75)
        cubic_x = distance_axis[:split_index]
        linear_x = distance_axis[split_index - 1 :]
        cubic_interpolator = interp1d(
            x, y, kind="linear", bounds_error=False, fill_value="extrapolate"
        )
        linear_interpolator = interp1d(
            x, y, kind="linear", bounds_error=False, fill_value="extrapolate"
        )
        cubic_y = cubic_interpolator(cubic_x)
        linear_y = linear_interpolator(linear_x)
        combined_y = np.concatenate((cubic_y, linear_y[1:]))
        interpolated_forces.append(combined_y)
    return interpolated_forces


# Calculate mean curve and statistical envelopes
def calculate_group_envelopes(forces, sigma_multiplier):
    forces_array = np.array(forces)
    mean_force = np.mean(forces_array, axis=0)
    std_dev_force = np.std(forces_array, axis=0)
    upper_envelope = mean_force + sigma_multiplier * std_dev_force
    lower_envelope = mean_force - sigma_multiplier * std_dev_force
    return mean_force, upper_envelope, lower_envelope


# Sample envelope coordinates for table generation
def sample_coordinates(distance_axis, envelope, num_points):
    curve_indexs = np.linspace(0, len(distance_axis) - 1, num_points, dtype=int)
    sampled_coords = pd.DataFrame(
        {
            "Distance (mm)": distance_axis[curve_indexs],
            "Envelope (kN)": envelope[curve_indexs],
        }
    )
    return sampled_coords


def print_sampled_coordinates_as_table(sampled_coords_min, sampled_coords_max):
    # Sort upper and lower envelope coordinate tables by distance
    sampled_coords_min_sorted = sampled_coords_min.sort_values(
        by="Distance (mm)"
    ).reset_index(drop=True)
    sampled_coords_max_sorted = sampled_coords_max.sort_values(
        by="Distance (mm)"
    ).reset_index(drop=True)

    # Table header
    print(f"At {sigma_slider.value} sigmas")
    print("Distance (mm)\tUpper Envelope (kN)\tDistance (mm)\tLower Envelope (kN)")

    # Iterate over rows using their index
    for i in range(min(len(sampled_coords_min_sorted), len(sampled_coords_max_sorted))):
        upper_row = sampled_coords_min_sorted.iloc[i]
        lower_row = sampled_coords_max_sorted.iloc[i]

        print(
            f"{upper_row['Distance (mm)']}\t{upper_row['Envelope (kN)']}\t{lower_row['Distance (mm)']}\t{lower_row['Envelope (kN)']}"
        )


# File upload logic
def handle_file_selection(b):
    curve_type = curve_type_buttons.value
    press_type = press_type_buttons.value

    delete_everything_in_content()

    print(f"Selected: {curve_type} + {press_type}")

    if curve_type == "Production" and press_type == "Promess":
        print("Please upload ONE file.")
        uploaded = files.upload()
        if uploaded:
            file_name = next(iter(uploaded))
            os.makedirs(
                "Production", exist_ok=True
            )  # Create folder if it doesn't exist
            new_path = os.path.join("Production", file_name)
            os.replace(file_name, new_path)  # Move the uploaded file
            print("✅ File saved to:", os.path.abspath(new_path))
            print(new_path)
        global distances, forces
        distances, forces = read_data_promess_xlsx(new_path)
        display_plot()

    elif curve_type == "Maximum-Minimum" and press_type == "Promess":
        print("Please upload MAXIMUM file.")
        uploaded = files.upload()
        if uploaded:
            file_name = next(iter(uploaded))
            os.makedirs("Maximum", exist_ok=True)  # Create folder if it doesn't exist
            maximum_path = os.path.join("Maximum", file_name)
            os.replace(file_name, maximum_path)  # Move the uploaded file
            print("✅ File saved to:", os.path.abspath(maximum_path))
        global distances_min, forces_min, distances_max, forces_max
        distances_max, forces_max = read_data_promess_xlsx(maximum_path)

        print("Please upload MINIMUM file.")
        uploaded = files.upload()
        if uploaded:
            file_name = next(iter(uploaded))
            os.makedirs("Minimum", exist_ok=True)  # Create folder if it doesn't exist
            minimum_path = os.path.join("Minimum", file_name)
            os.replace(file_name, minimum_path)  # Move the uploaded file
            print("✅ File saved to:", os.path.abspath(minimum_path))
        distances_min, forces_min = read_data_promess_xlsx(minimum_path)
        display_plot()

    elif curve_type == "Production" and press_type == "Kistler":
        print("Please upload MULTIPLE files from the same folder.")
        uploaded = files.upload()
        if uploaded:
            folder_name = "Production"
            os.makedirs(folder_name, exist_ok=True)
            for file_name in uploaded:
                destination = os.path.join(folder_name, file_name)
                os.replace(file_name, destination)
            production_folder_path = os.path.abspath(folder_name)
            print("✅ Files saved to:", production_folder_path)
        distances, forces = read_data_kistler_csv(production_folder_path, ",", ".")
        display_plot()

    elif curve_type == "Maximum-Minimum" and press_type == "Kistler":
        print("Please upload Maximum files.")
        uploaded = files.upload()
        if uploaded:
            file_name = next(iter(uploaded))
            os.makedirs("Maximum", exist_ok=True)  # Create folder if it doesn't exist
            for file_name in uploaded:
                destination = os.path.join("Maximum", file_name)
                os.replace(file_name, destination)
            maximum_folder_path = os.path.abspath("Maximum")
            print("✅ Files saved to:", maximum_folder_path)
        distances_max, forces_max = read_data_kistler_csv(maximum_folder_path, ",", ".")

        print("Please upload Minimum files.")
        uploaded = files.upload()
        if uploaded:
            file_name = next(iter(uploaded))
            os.makedirs("Minimum", exist_ok=True)  # Create folder if it doesn't exist
            for file_name in uploaded:
                destination = os.path.join("Minimum", file_name)
                os.replace(file_name, destination)
            minimum_folder_path = os.path.abspath("Minimum")
            print("✅ Files saved to:", maximum_folder_path)
        distances_min, forces_min = read_data_kistler_csv(minimum_folder_path, ",", ".")
        display_plot()
    else:
        print("⚠️ Unknown selection.")


# Generate and display the interactive plot
def display_plot():
    curve_type, press_type, resolution, sigma_multiplier, num_points_table = (
        curve_type_buttons.value,
        press_type_buttons.value,
        resolution_slider.value,
        sigma_slider.value,
        num_points_slider.value,
    )

    if curve_type == "Production":
        min_distance = min(map(max, distances))
        max_distance = max(map(min, distances))
    elif curve_type == "Maximum-Minimum":
        min_distance = min(map(max, distances_min + distances_max))
        max_distance = max(map(min, distances_min + distances_max))

    # Generar eje de distancias
    distance_axis = np.linspace(min_distance, max_distance, resolution)

    # Interpolar curvas
    if curve_type == "Production":
        forces_mixed = interpolate_mixed(distances, forces, distance_axis)
    elif curve_type == "Maximum-Minimum":
        forces_min_mixed = interpolate_mixed(distances_min, forces_min, distance_axis)
        forces_max_mixed = interpolate_mixed(distances_max, forces_max, distance_axis)

    # Calcular envolventes
    if curve_type == "Production":
        mean, upper_envelope, lower_envelope = calculate_group_envelopes(
            forces_mixed, sigma_multiplier
        )
    elif curve_type == "Maximum-Minimum":
        mean_min, upper_envelope, _ = calculate_group_envelopes(
            forces_min_mixed, sigma_multiplier
        )
        mean_max, _, lower_envelope = calculate_group_envelopes(
            forces_max_mixed, sigma_multiplier
        )

    # Muestrear coordenadas para la tabla
    sampled_coords_min = sample_coordinates(
        distance_axis, upper_envelope, num_points_table
    )
    sampled_coords_max = sample_coordinates(
        distance_axis, lower_envelope, num_points_table
    )

    fig = go.Figure()

    # Add envelopes
    fig.add_trace(
        go.Scatter(
            x=distance_axis,
            y=upper_envelope,
            mode="lines",
            line=dict(color="green", width=4),
            name="Upper Envelope",
            hovertemplate="Distance: %{x}<br>Upper Envelope: %{y}<extra></extra>",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=distance_axis,
            y=lower_envelope,
            mode="lines",
            line=dict(color="green", width=4),
            name="Lower Envelope",
            hovertemplate="Distance: %{x}<br>Lower Envelope: %{y}<extra></extra>",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=sampled_coords_min["Distance (mm)"],
            y=sampled_coords_min["Envelope (kN)"],
            mode="markers",
            marker=dict(
                color="green", size=10
            ),  # Optional: size and color of the point
            name="Sampled upper envelope curve",
            hovertemplate="Distance: %{x}<br>Lower Envelope: %{y}<extra></extra>",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=sampled_coords_max["Distance (mm)"],
            y=sampled_coords_max["Envelope (kN)"],
            mode="markers",
            marker=dict(
                color="green", size=10
            ),  # Optional: size and color of the point
            name="Sampled lower envelope curve",
            hovertemplate="Distance: %{x}<br>Lower Envelope: %{y}<extra></extra>",
        )
    )

    if curve_type == "Production":
        for i, (x, y) in enumerate(zip(distances, forces)):
            fig.add_trace(
                go.Scatter(
                    x=x,
                    y=y,
                    mode="lines",
                    line=dict(color="blue", width=width_line.value),
                    name=f"Curve {i + 1}",  # Curve identifier
                    hovertemplate=f"Curve: {i + 1}<br>Distance: %{{x}}<br>Force: %{{y}}<extra></extra>",
                    customdata=[i] * len(x),  # Pass curve index
                )
            )
        fig.add_trace(
            go.Scatter(
                x=distance_axis,
                y=mean,
                mode="lines",
                line=dict(color="black", width=4),
                name="Mean Curve",
                hovertemplate="Distance: %{x}<br>Mean Force: %{y}<extra></extra>",
            )
        )
        title = (
            f"Envelope Curves Generated from Production Data ({sigma_multiplier:.0f}σ)"
        )
    elif curve_type == "Maximum-Minimum":
        for i, (x, y) in enumerate(zip(distances_max, forces_max)):
            fig.add_trace(
                go.Scatter(
                    x=x,
                    y=y,
                    mode="lines",
                    line=dict(color="red", width=width_line.value),
                    name=f"Curve {i + 1}",  # Curve identifier
                    hovertemplate=f"Curve: {i + 1}<br>Distance: %{{x}}<br>Force: %{{y}}<extra></extra>",
                    customdata=[i] * len(x),  # Pass curve index
                )
            )
        fig.add_trace(
            go.Scatter(
                x=distance_axis,
                y=mean_max,
                mode="lines",
                line=dict(color="black", width=4),
                name="Mean Curve max",
                hovertemplate="Distance: %{x}<br>Mean Force: %{y}<extra></extra>",
            )
        )
        for i, (x, y) in enumerate(zip(distances_min, forces_min)):
            fig.add_trace(
                go.Scatter(
                    x=x,
                    y=y,
                    mode="lines",
                    line=dict(color="blue", width=width_line.value),
                    name=f"Curve {i + 1 + len(distances_max)}",  # Curve identifier
                    hovertemplate=f"Curve: {i + 1 + len(distances_max)}<br>Distance: %{{x}}<br>Force: %{{y}}<extra></extra>",
                    customdata=[i] * len(x),  # Pass curve index
                )
            )
        fig.add_trace(
            go.Scatter(
                x=distance_axis,
                y=mean_min,
                mode="lines",
                line=dict(color="black", width=4),
                name="Mean Curve max",
                hovertemplate="Distance: %{x}<br>Mean Force: %{y}<extra></extra>",
            )
        )
        title = f"Envelope Curves Generated from Maximum/Minimum Data ({sigma_multiplier:.0f}σ)"

    # Update layout for interactivity
    fig.update_layout(
        title=title,
        xaxis_title="Distance (mm)",
        yaxis_title="Force (kN)",
        hovermode="closest",  # Show nearest point on hover
        template="plotly_white",
        dragmode="pan",
        width=1200,  # Wider figure
        height=600,  # Adjust height if needed
    )

    clear_output(wait=True)  # Clear previous output to avoid duplicate rendering
    display(
        resolution_slider,
        sigma_slider,
        num_points_slider,
        width_line,
        curve_delete_text,
        delete_curve_button,
        del_last_point_button,
        fig,
    )  # Vuelve a mostrar la figura y el botón
    print_sampled_coordinates_as_table(sampled_coords_min, sampled_coords_max)


# Callback function triggered when the button is clicked
def update_plot(b):
    display_plot()


def delete_curves(b):
    curve_type = (
        curve_type_buttons.value
    )  # Ensure curve_type is set to either "Production" or "Maximum-Minimum"

    curve_index = curve_delete_text.value
    curve_index_str = str(curve_index).strip()

    if "," in curve_index_str:
        parts = [p.strip() for p in curve_index_str.split(",")]
        try:
            curve_indexs = list(map(int, parts))
        except ValueError:
            print("Error: One or more values are not valid integers.")
            return
    else:
        try:
            curve_indexs = [int(curve_index_str)]
        except ValueError:
            print("Error: The entered value is not an integer.")
            return

    curve_indexs = sorted(curve_indexs, reverse=True)

    if curve_type == "Production":
        for i in curve_indexs:
            if 0 < i <= len(distances):
                del distances[i - 1]
                del forces[i - 1]
                print(f"[Production] Curve {i} deleted successfully.")
            else:
                print(f"[Production] Index out of range: {i}")

    elif curve_type == "Maximum-Minimum":
        total_max = len(distances_max)
        total_min = len(distances_min)
        total = total_max + total_min

        for i in curve_indexs:
            if i < 1 or i > total:
                print(f"[Max-Min] Index out of range: {i}")
                continue

            i0 = i - 1  # Adjust to zero-based indexing

            if i0 < total_max:
                del distances_max[i0]
                del forces_max[i0]
                print(f"[Max] Curve {i} deleted successfully (local index: {i0}).")
            else:
                i_min = i0 - total_max
                if i_min < total_min:
                    del distances_min[i_min]
                    del forces_min[i_min]
                    print(
                        f"[Min] Curve {i} deleted successfully (local index: {i_min})."
                    )
                else:
                    print(f"[Minimum] Index out of range: {i}")

    else:
        print("Unknown curve type. Expected 'Production' or 'Maximum-Minimum'.")

    display_plot()


# Delete the lastest points of the each curve
def delete_last_points(b):
    global distances, forces, distances_min, forces_min, distances_max, forces_max
    curve_type = curve_type_buttons.value

    if curve_type == "Production":
        for i in range(len(distances)):
            distances[i] = distances[i][:-1]
            forces[i] = forces[i][:-1]
    elif curve_type == "Maximum-Minimum":
        for i in range(len(distances_max)):
            distances_max[i] = distances_max[i][:-1]
            forces_max[i] = forces_max[i][:-1]
        for i in range(len(distances_min)):
            distances_min[i] = distances_min[i][:-1]
            forces_min[i] = forces_min[i][:-1]
    display_plot()


# Toggle Buttons
curve_type_buttons = widgets.ToggleButtons(
    options=["Production", "Maximum-Minimum"],
    description="Curve type:",
)

press_type_buttons = widgets.ToggleButtons(
    options=["Promess", "Kistler"], description="Press type:"
)

# Sliders
resolution_slider = widgets.IntSlider(
    min=1, max=500, step=10, value=100, description="Resolution:"
)
sigma_slider = widgets.FloatSlider(
    min=1, max=10, step=0.5, value=6, description="Sigma Multiplier:"
)
num_points_slider = widgets.IntSlider(
    min=5, max=100, step=5, value=30, description="Table Points:"
)
width_line = widgets.FloatSlider(
    min=0.1, max=5, step=0.1, value=0.2, description="Curve width:"
)
curve_delete_text = widgets.Text(
    value="157 ó 101,159,200",
    placeholder="Example: 157 or 101,159,200",
    description="Curva a borrar",
    disabled=False,
)
delete_curve_button = widgets.Button(description="Delete Curve(s)")
del_last_point_button = widgets.Button(description="Delete Last Point")

# Buttons
boton = widgets.Button(description="Refresh Plot")

# Button to trigger upload
upload_button = widgets.Button(
    description="Upload Files",
    button_style="info",
    tooltip="Click to upload",
    icon="upload",
)

# Button function
resolution_slider.observe(update_plot)
sigma_slider.observe(update_plot)
num_points_slider.observe(update_plot)
boton.on_click(update_plot)
upload_button.on_click(handle_file_selection)
width_line.observe(update_plot)
delete_curve_button.on_click(delete_curves)
del_last_point_button.on_click(delete_last_points)

# Display graph and buttons
display(curve_type_buttons, press_type_buttons, upload_button)


IntSlider(value=100, description='Resolution:', max=500, min=1, step=10)

FloatSlider(value=6.0, description='Sigma Multiplier:', max=10.0, min=1.0, step=0.5)

IntSlider(value=30, description='Table Points:', min=5, step=5)

FloatSlider(value=0.2, description='Curve width:', max=5.0, min=0.1)

Text(value='157 ó 101,159,200', description='Curva a borrar', placeholder='Example: 157 or 101,159,200')

Button(description='Delete Curve(s)', style=ButtonStyle())

Button(description='Delete Last Point', style=ButtonStyle())

At 6.0 sigmas
Distance (mm)	Upper Envelope (kN)	Distance (mm)	Lower Envelope (kN)
157.05130004882812	0.1507882952882461	157.05130004882812	0.032076974994405094
157.15879528931896	0.15806031316232524	157.15879528931896	0.038528777295373046
157.2394167196871	0.16093426653594212	157.2394167196871	0.04343577562536316
157.34691196017795	0.16830832740697035	157.34691196017795	0.04743673206867987
157.42753339054607	0.17455596586312716	157.42753339054607	0.055298241413135114
157.53502863103694	0.1901373002917462	157.53502863103694	0.060260794527521755
157.61565006140506	0.2196212520376083	157.61565006140506	0.06089719432971123
157.6962714917732	0.3894095137218281	157.6962714917732	0.0059304009026457105
157.80376673226405	0.8334986668504586	157.80376673226405	-0.03553713659614244
157.88438816263218	1.212259432000263	157.88438816263218	0.062336815193699824
157.99188340312304	1.7539501019617665	157.99188340312304	0.28608556279654684
158.07250483349117	2.2018641299087536	158.07250483349117	0.48420